# Week 3 — Phase A2: Architecture Overhaul

## Strategy
DigiSteel v3: YOLOv11s + standard Conv + CoordAttention at P3 + Inner-WIoU.
Removed: GhostConv, WFCA (2×), EMA (4×).

## What Changed vs v2
| Component | v2 (old) | v3 (new) | Why |
|-----------|----------|----------|-----|
| Backbone Conv | GhostConv | **Standard Conv** | Ghost halved spatial filtering—hurts texture defects |
| WFCA | 2× (P2, P3) | **Removed** | DWT on 25×25 maps loses resolution; +0.1% gain |
| EMA neck | 4× | **Removed** | Attention stacking—4 gates before network learns |
| New attention | None | **CA at P3 only** | Preserves spatial coords via directional pooling |
| Model scale | YOLOv11n (2.6M) | **YOLOv11s (~9.4M)** | 2× backbone for fine features |
| Inner-WIoU | Yes | **Keep** | Literature-supported, not the bottleneck |
| ImgSz | 800 | **800** | 17GB VRAM — keep 800 even for v11s |

## Expected
- **A1 result:** 78-79% mAP@0.5
- **A2 expected:** 80-82% mAP@0.5
- **Train time:** ~8 hours (larger model + more epochs)

## Setup & Register Custom Modules

In [ ]:
%matplotlib inline
import torch
import json
import sys
from pathlib import Path
from ultralytics import YOLO
import warnings
warnings.filterwarnings("ignore")

# Project root
PROJECT_ROOT = Path(r"D:\DigiSteel-Yolo\DigiSteel-YOLO")
sys.path.insert(0, str(PROJECT_ROOT))

# Paths
DATA_YAML = str(PROJECT_ROOT / "configs" / "data" / "neu_det.yaml")
MODEL_YAML = str(PROJECT_ROOT / "configs" / "models" / "digisteel_v3.yaml")
EVALS_DIR = PROJECT_ROOT / "evals"
EVALS_DIR.mkdir(exist_ok=True)

# Register CoordAttention (must be BEFORE loading YAML)
import ultralytics.nn.tasks as tasks
from digisteel.modules import CoordAttention
tasks.CoordAttention = CoordAttention

# Register ALL modules for compatibility (trainer needs them for Inner-WIoU)
from digisteel.engine.trainer import register_custom_modules, DigiSteelTrainer
register_custom_modules()

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({vram_total:.1f} GB VRAM)")

## Preview Architecture

In [ ]:
# Quick preview — build model and check it compiles
print("=== DigiSteel v3 Architecture ===")
print(f"Config: {MODEL_YAML}")
print()

# Build without pretrained to check structure
temp_model = YOLO(MODEL_YAML)
info = temp_model.model
num_params = sum(p.numel() for p in info.parameters()) / 1e6
num_layers = len(list(info.modules()))

# Count CA and other custom modules
ca_count = sum(1 for m in info.modules() if isinstance(m, CoordAttention))

print(f"Layers: {num_layers}")
print(f"Parameters: {num_params:.2f}M")
print(f"CoordAttention modules: {ca_count}")
print()
print("Architecture check: PASS" if ca_count == 1 else f"WARNING: Expected 1 CA, found {ca_count}")
del temp_model  # free memory

## GPU Memory Check

In [ ]:
print("GPU memory before training:")
!nvidia-smi --query-gpu=memory.used,memory.free,memory.total --format=csv

## Train — DigiSteel v3

Uses DigiSteelTrainer for Inner-WIoU injection.
Augmentation mirrors Phase A1 config.
17GB VRAM — can keep imgsz=800 even for YOLOv11s.

In [ ]:
# ---- Week 3 A2: Architecture Fix ----

overrides = {
    "model": MODEL_YAML,
    "data": DATA_YAML,
    "epochs": 400,
    "imgsz": 800,           # 17GB VRAM — keep 800 even for v11s
    "batch": 16,            # YOLOv11s at 800px — safe for 17GB\n    "seed": 42,
    "project": str(PROJECT_ROOT / "runs"),
    "name": "week3_a2_digisteel_v3",
    "exist_ok": True,
    "patience": 100,
    "amp": True,
    "cos_lr": True,
    "close_mosaic": 0,
    "verbose": True,
    "pretrained": True,     # use YOLOv11s pretrained weights
    
    # --- Augmentation (same as A1) ---
    "mosaic": 0.0,
    "mixup": 0.15,
    "degrees": 10.0,
    "translate": 0.2,
    "scale": 0.6,
    "shear": 5.0,
    "fliplr": 0.5,
    "flipud": 0.0,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "copy_paste": 0.0,
}

print("Training DigiSteel v3 (YOLOv11s + CA + Inner-WIoU)...")
print(f"  Model: digisteel_v3.yaml")
print(f"  ImgSz: {overrides['imgsz']}, Batch: {overrides['batch']}")
print(f"  Augmentation: mosaic=0, mixup=0.15, degrees=10, shear=5")
print(f"  Epochs: {overrides['epochs']}, Patience: {overrides['patience']}")

trainer = DigiSteelTrainer.create_trainer(overrides)
trainer.train()

weights_path = str(PROJECT_ROOT / "runs" / "week3_a2_digisteel_v3" / "weights" / "best.pt")
print(f"\nTraining complete!")
print(f"Best weights: {weights_path}")

## Evaluate

In [ ]:
# Fresh evaluation on validation set
weights_path = str(PROJECT_ROOT / "runs" / "week3_a2_digisteel_v3" / "weights" / "best.pt")
model = YOLO(weights_path)
metrics = model.val(data=DATA_YAML, split='test', imgsz=800, verbose=True)

# Extract metrics
mp_val = float(metrics.box.mp) if metrics.box.mp is not None else 0.0
mr_val = float(metrics.box.mr) if metrics.box.mr is not None else 0.0

results_dict = {
    "phase": "Week3-A2",
    "description": "Architecture fix — YOLOv11s + Conv (no Ghost) + CA at P3 + Inner-WIoU",
    "mAP50": round(float(metrics.box.map50), 4),
    "mAP50_95": round(float(metrics.box.map), 4),
    "precision": round(mp_val, 4),
    "recall": round(mr_val, 4),
    "f1": round(2 * mp_val * mr_val / (mp_val + mr_val + 1e-7), 4),
}

print(f"\n=== Week 3 A2 Results ===")
print(f"mAP@0.5:       {results_dict['mAP50']*100:.1f}%")
print(f"mAP@0.5:0.95:  {results_dict['mAP50_95']*100:.1f}%")
print(f"Precision:     {results_dict['precision']*100:.1f}%")
print(f"Recall:        {results_dict['recall']*100:.1f}%")
print(f"F1:            {results_dict['f1']*100:.1f}%")

## Per-Class Analysis

In [ ]:
# Per-class mAP breakdown (safe API for ultralytics 8.3.x)
class_names = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

try:
    per_class_ap50 = {}
    # Try maps first (safe in 8.3.x), then fall back to ap_class_index + ap50
    if hasattr(metrics.box, 'maps') and metrics.box.maps is not None:
        maps_list = metrics.box.maps.tolist() if hasattr(metrics.box.maps, 'tolist') else list(metrics.box.maps)
        for i, ap in enumerate(maps_list):
            if i < len(class_names):
                per_class_ap50[class_names[i]] = round(float(ap), 4)
    elif hasattr(metrics.box, 'ap_class_index'):
        for i, cls_id in enumerate(metrics.box.ap_class_index):
            if cls_id < len(class_names):
                per_class_ap50[class_names[cls_id]] = round(float(metrics.box.ap50[i]), 4)

    if per_class_ap50:
        print("Per-class mAP@0.5:")
        for name in class_names:
            ap = per_class_ap50.get(name, 0)
            bar = "█" * int(ap * 50)
            print(f"  {name:20s}: {ap*100:5.1f}%  {bar}")
        results_dict["per_class_mAP50"] = per_class_ap50
    else:
        print("Per-class AP not available.")
        print("Run model.val(split='test') separately for per-class breakdown.")
except Exception as e:
    print(f"Per-class AP extraction failed: {e}")
    print("Run model.val(split='test') separately for per-class breakdown.")

## Save & Compare

In [ ]:
# Save to evals
results_path = EVALS_DIR / "week3_a2_results.json"
with open(results_path, "w") as f:
    json.dump(results_dict, f, indent=2)

print(f"Results saved to: {results_path}")

# Compare with A1
a1_path = EVALS_DIR / "week3_a1_results.json"
if a1_path.exists():
    with open(a1_path) as f:
        a1_results = json.load(f)
    a1_map50 = a1_results.get("mAP50", 0)
else:
    a1_map50 = 0.758  # fallback to Week 2 baseline
    print("(A1 results not found, comparing against Week 2 baseline)")

print(f"\n=== Week 3 Summary ===")
print(f"A1 (Config Fix):     {a1_map50*100:.1f}% mAP@0.5")
print(f"A2 (Arch Fix):       {results_dict['mAP50']*100:.1f}% mAP@0.5")
print(f"Week 3 Total Gain:   {(results_dict['mAP50'] - 0.758)*100:+.1f}% from Week 2 baseline")
print(f"\nDone!")